# A* Search - From Dijkstra to Heuristic Shortest Paths

In this notebook we will start with a short refresher on Dijkstra's algorithm and then move on to A* search.

Goals for today:

* Review shortest path ideas from Dijkstra's algorithm
* Understand the role of a heuristic function in A*
* See why `f(n) = g(n) + h(n)` is the key formula
* Build a Python implementation of A*
* Visualize the algorithm step by step on a grid


In [ ]:
import sys
import math
import heapq
import numpy as np
import matplotlib.pyplot as plt

print(f"Int Max is {sys.maxsize} - this will be our infinity")


## Problem Setup

We will use a small grid world where:

* `0` means a free cell
* `1` means a wall / obstacle

This is a nice example because Dijkstra and A* can be compared on exactly the same search space.


In [ ]:
# 0 = free cell, 1 = wall
grid = np.array([
    [0, 0, 0, 0, 0, 0],
    [0, 1, 1, 1, 0, 0],
    [0, 0, 0, 1, 0, 0],
    [0, 1, 0, 0, 0, 1],
    [0, 1, 0, 1, 0, 0],
    [0, 0, 0, 0, 0, 0],
])

start = (0, 0)
goal = (5, 5)

grid


In [ ]:
def draw_grid(grid, start=None, goal=None, path=None, expanded=None, frontier=None, title=""):
    rows, cols = grid.shape
    canvas = np.ones((rows, cols, 3))

    # walls are black
    for r in range(rows):
        for c in range(cols):
            if grid[r, c] == 1:
                canvas[r, c] = [0.1, 0.1, 0.1]

    # expanded nodes are light blue
    if expanded:
        for r, c in expanded:
            if (r, c) != start and (r, c) != goal and grid[r, c] == 0:
                canvas[r, c] = [0.7, 0.85, 1.0]

    # frontier nodes are orange
    if frontier:
        for r, c in frontier:
            if (r, c) != start and (r, c) != goal and grid[r, c] == 0:
                canvas[r, c] = [1.0, 0.85, 0.55]

    # final path is green
    if path:
        for r, c in path:
            if (r, c) != start and (r, c) != goal:
                canvas[r, c] = [0.5, 0.85, 0.5]

    # start is blue and goal is red
    if start:
        canvas[start] = [0.3, 0.5, 1.0]
    if goal:
        canvas[goal] = [1.0, 0.35, 0.35]

    plt.figure(figsize=(6, 6))
    plt.imshow(canvas)
    plt.xticks(range(cols))
    plt.yticks(range(rows))
    plt.grid(True)
    plt.title(title)
    plt.show()


In [ ]:
draw_grid(grid, start=start, goal=goal, title="Initial Grid")


## Dijkstra Algorithm Refresher

Dijkstra's algorithm finds shortest paths in a graph with non-negative edge weights.

Main idea:

* Keep the best known distance from the start to each node
* Always expand the frontier node with smallest known distance
* Relax outgoing edges and update distances when a shorter route is found

The key value used by Dijkstra is:

* `g(n)` = exact cost from the start node to node `n`


## Dijkstra Pseudocode

```text
DIJKSTRA(G, start, goal):
    dist[start] = 0
    dist[all other nodes] = infinity
    prev[node] = None
    priority queue = [(0, start)]

    while priority queue not empty:
        (current_dist, u) = extract_min()

        if u == goal:
            break

        if current_dist > dist[u]:
            continue

        for each neighbor v of u with edge cost w:
            alt = dist[u] + w
            if alt < dist[v]:
                dist[v] = alt
                prev[v] = u
                push (alt, v) into priority queue

    return dist, prev
```


In [ ]:
def in_bounds(grid, node):
    r, c = node
    return 0 <= r < grid.shape[0] and 0 <= c < grid.shape[1]

def passable(grid, node):
    r, c = node
    return grid[r, c] == 0

def neighbors_4(grid, node):
    r, c = node
    candidates = [(r - 1, c), (r + 1, c), (r, c - 1), (r, c + 1)]
    return [p for p in candidates if in_bounds(grid, p) and passable(grid, p)]

def reconstruct_path(came_from, start, goal):
    if goal not in came_from and goal != start:
        return None

    path = [goal]
    current = goal
    while current != start:
        current = came_from[current]
        path.append(current)
    path.reverse()
    return path


In [ ]:
# let's implement Dijkstra on our grid using a priority queue
def dijkstra_search(grid, start, goal, record_steps=False):
    pq = [(0, start)]
    g_score = {start: 0}
    came_from = {}
    expanded = set()
    snapshots = []

    while pq:
        current_cost, current = heapq.heappop(pq)

        if current in expanded:
            continue

        expanded.add(current)

        if record_steps:
            frontier_nodes = [node for _, node in pq]
            snapshots.append({
                "current": current,
                "expanded": set(expanded),
                "frontier": set(frontier_nodes),
                "g_score": dict(g_score),
                "came_from": dict(came_from),
            })

        if current == goal:
            path = reconstruct_path(came_from, start, goal)
            return {
                "path": path,
                "cost": g_score[goal],
                "expanded": expanded,
                "snapshots": snapshots,
            }

        for nxt in neighbors_4(grid, current):
            tentative = g_score[current] + 1
            if tentative < g_score.get(nxt, math.inf):
                g_score[nxt] = tentative
                came_from[nxt] = current
                heapq.heappush(pq, (tentative, nxt))

    return {
        "path": None,
        "cost": math.inf,
        "expanded": expanded,
        "snapshots": snapshots,
    }


In [ ]:
dijkstra_result = dijkstra_search(grid, start, goal, record_steps=True)
print("Dijkstra shortest path cost:", dijkstra_result["cost"])
print("Dijkstra path:", dijkstra_result["path"])


In [ ]:
draw_grid(
    grid,
    start=start,
    goal=goal,
    path=dijkstra_result["path"],
    expanded=dijkstra_result["expanded"],
    title="Dijkstra Result"
)


## Why Move Beyond Dijkstra?

Dijkstra's algorithm is correct and reliable, but it does not know where the goal is.

So on a map or grid it often explores many nodes in directions that do not help us reach the goal quickly.

This motivates the idea of a heuristic.


## A* Algorithm - Using Heuristic Function to find the shortest path

* Key idea - Use a heuristic function to estimate the cost of the cheapest path from the start to the goal.
* `g(n)` is the exact cost from the start to node `n`.
* `h(n)` is the estimated cost from node `n` to the goal.
* `f(n) = g(n) + h(n)` is the estimated total cost of a path through node `n`.
* If you do not use a heuristic function, A* is equivalent to Dijkstra's algorithm.
* If your heuristic function is not admissible, A* is not guaranteed to find the shortest path.


## A* Pseudocode

```text
A_STAR(G, start, goal, h):
    g[start] = 0
    f[start] = h(start, goal)
    prev[node] = None
    priority queue = [(f[start], start)]

    while priority queue not empty:
        (_, u) = extract_min()

        if u == goal:
            return path

        for each neighbor v of u with edge cost w:
            tentative_g = g[u] + w
            if tentative_g < g[v]:
                prev[v] = u
                g[v] = tentative_g
                f[v] = g[v] + h(v, goal)
                push (f[v], v) into priority queue

    return failure
```


In [ ]:
# here are some heuristic functions we can use on the grid
def zero_heuristic(a, b):
    return 0

def manhattan(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def euclidean(a, b):
    return math.hypot(a[0] - b[0], a[1] - b[1])


In [ ]:
# let's implement A* algorithm - it is a modification of Dijkstra algorithm
def a_star_search(grid, start, goal, heuristic=manhattan, record_steps=False):
    pq = [(heuristic(start, goal), start)]
    g_score = {start: 0}
    came_from = {}
    expanded = set()
    snapshots = []

    while pq:
        current_f, current = heapq.heappop(pq)

        if current in expanded:
            continue

        expanded.add(current)

        if record_steps:
            frontier_nodes = [node for _, node in pq]
            snapshots.append({
                "current": current,
                "expanded": set(expanded),
                "frontier": set(frontier_nodes),
                "g_score": dict(g_score),
                "came_from": dict(came_from),
            })

        if current == goal:
            path = reconstruct_path(came_from, start, goal)
            return {
                "path": path,
                "cost": g_score[goal],
                "expanded": expanded,
                "snapshots": snapshots,
            }

        for nxt in neighbors_4(grid, current):
            tentative_g = g_score[current] + 1
            if tentative_g < g_score.get(nxt, math.inf):
                came_from[nxt] = current
                g_score[nxt] = tentative_g
                f_score = tentative_g + heuristic(nxt, goal)
                heapq.heappush(pq, (f_score, nxt))

    return {
        "path": None,
        "cost": math.inf,
        "expanded": expanded,
        "snapshots": snapshots,
    }


In [ ]:
astar_result = a_star_search(grid, start, goal, heuristic=manhattan, record_steps=True)
print("A* shortest path cost:", astar_result["cost"])
print("A* path:", astar_result["path"])


In [ ]:
draw_grid(
    grid,
    start=start,
    goal=goal,
    path=astar_result["path"],
    expanded=astar_result["expanded"],
    title="A* Result using Manhattan Distance"
)


## Dijkstra vs A*

Now let's compare both algorithms on the same grid.


In [ ]:
print("Dijkstra cost:", dijkstra_result["cost"])
print("A* cost:", astar_result["cost"])
print("Dijkstra expanded nodes:", len(dijkstra_result["expanded"]))
print("A* expanded nodes:", len(astar_result["expanded"]))


## Step by Step Visualization

To understand search algorithms properly, it is useful to inspect the process and not only the final answer.

Each snapshot stores:

* current node
* expanded nodes
* frontier nodes
* current best known distances
* predecessor links for path reconstruction


In [ ]:
def show_snapshot(grid, snapshots, step, start, goal, title_prefix=""):
    snap = snapshots[step]
    draw_grid(
        grid,
        start=start,
        goal=goal,
        expanded=snap["expanded"],
        frontier=snap["frontier"],
        title=f"{title_prefix} Step {step}: current={snap['current']}"
    )


In [ ]:
# inspect one step from each algorithm
show_snapshot(grid, dijkstra_result["snapshots"], step=0, start=start, goal=goal, title_prefix="Dijkstra")
show_snapshot(grid, astar_result["snapshots"], step=0, start=start, goal=goal, title_prefix="A*")


## Optional Interactive Exploration

If `ipywidgets` is available, you can browse the snapshots using a slider.


In [ ]:
try:
    from ipywidgets import interact, IntSlider
except ImportError:
    interact = None
    IntSlider = None
    print("ipywidgets is not installed - skipping interactive slider")


In [ ]:
if interact is not None:
    @interact(step=IntSlider(min=0, max=len(astar_result["snapshots"]) - 1, step=1, value=0))
    def browse_astar(step=0):
        show_snapshot(grid, astar_result["snapshots"], step, start, goal, title_prefix="A*")


## Experiments

First, if we set the heuristic to 0, then A* should behave like Dijkstra.


In [ ]:
astar_zero = a_star_search(grid, start, goal, heuristic=zero_heuristic, record_steps=False)
print("A* with h=0 cost:", astar_zero["cost"])
print("A* with h=0 expanded nodes:", len(astar_zero["expanded"]))


In [ ]:
for h_name, h_fn in [
    ("zero", zero_heuristic),
    ("manhattan", manhattan),
    ("euclidean", euclidean),
]:
    result = a_star_search(grid, start, goal, heuristic=h_fn, record_steps=False)
    print(f"{h_name:>10} | cost={result['cost']} | expanded={len(result['expanded'])}")


## Bad Heuristic Example

Here we intentionally use a heuristic that may overestimate.

This can make A* faster, but then optimality is no longer guaranteed.


In [ ]:
def inflated_manhattan(a, b):
    return 2.0 * manhattan(a, b)

bad_result = a_star_search(grid, start, goal, heuristic=inflated_manhattan, record_steps=False)
print("Inflated heuristic cost:", bad_result["cost"])
print("Inflated heuristic expanded nodes:", len(bad_result["expanded"]))
print("Inflated heuristic path:", bad_result["path"])


## External Library

NetworkX contains implementations of both Dijkstra and A*.

Install it if needed:

* `pip install networkx`

Documentation URLs:

* https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.shortest_paths.weighted.dijkstra_path.html
* https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.shortest_paths.astar.astar_path.html


## Exercises

1. Modify the code to allow diagonal movement.
2. Replace Manhattan distance with a heuristic suitable for diagonal movement.
3. Count how many times each algorithm pushes into the priority queue.
4. Draw predecessor arrows instead of only showing the final path.
5. Adapt the code from a grid to a general weighted graph.


## External References

Verified on 2026-04-17.

* CLRS, 4th edition, official MIT Press page: https://mitpress.mit.edu/9780262046305/introduction-to-algorithms/
* MIT OpenCourseWare, Search: Optimal, Branch and Bound, A*: https://ocw.mit.edu/courses/6-034-artificial-intelligence-fall-2010/resources/lecture-5-search-optimal-branch-and-bound-a/
* Berkeley CS188, Informed Search: https://inst.eecs.berkeley.edu/~cs188/textbook/search/informed.html
* Hart, Nilsson, Raphael (1968), original A* paper DOI: https://doi.org/10.1109/TSSC.1968.300136
* Bibliographic record for the original paper: https://cir.nii.ac.jp/crid/1363388844102264832?lang=en
* NetworkX dijkstra_path documentation: https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.shortest_paths.weighted.dijkstra_path.html
* NetworkX astar_path documentation: https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.shortest_paths.astar.astar_path.html
